這個區塊負責載入必備的套件與設定路徑，維持極致的輕量化。加入了 warnings 過濾器，用來屏蔽 GARCH 模型在優化過程中產生的無害收斂警告，保持終端機輸出乾淨。

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import norm
import warnings

from pygam import LinearGAM, s, te
from arch import arch_model
from sklearn.metrics import recall_score, precision_score, accuracy_score, f1_score
from itertools import combinations

# 隱藏 GARCH 模型無害的優化收斂警告
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', module='arch')

# -------------------------
# Configuration and Paths
# -------------------------
PROJECT_DIRECTORY = Path.cwd().parents[0]

這裡包含了數學運算與閾值設定的輔助函數。變數名稱已經全面展開，去除了所有令人困惑的縮寫。

In [3]:
# -------------------------
# Math & Threshold Helpers
# -------------------------
def calculate_garch_sigma_manually(
    residuals_array, omega_constant, alpha_coefficient, beta_coefficient, 
    last_residual_from_training, last_variance_from_training
):
    total_time_steps = len(residuals_array)
    conditional_variances = np.zeros(total_time_steps)
    current_residual = float(last_residual_from_training)
    current_variance = float(last_variance_from_training)

    for time_index in range(total_time_steps):
        next_variance = omega_constant + alpha_coefficient * (current_residual**2) + beta_coefficient * current_variance
        conditional_variances[time_index] = next_variance
        current_variance = next_variance
        current_residual = residuals_array[time_index]

    return np.sqrt(np.maximum(conditional_variances, 1e-12))

def calculate_naive_max3_recall(time_series_data, pm25_threshold=15, evaluation_year=2024):
    """(保留供未來動態閾值實驗用)"""
    time_series_float = time_series_data.astype(float)
    lag_dataframe = pd.DataFrame(index=time_series_float.index)
    
    lag_dataframe["lag_0"] = time_series_float
    lag_dataframe["lag_1"] = time_series_float.shift(1)
    lag_dataframe["lag_2"] = time_series_float.shift(2)
    lag_dataframe["target_t_plus_3"] = time_series_float.shift(-3)
    
    lag_dataframe = lag_dataframe.dropna()
    lag_dataframe = lag_dataframe[lag_dataframe.index.year == evaluation_year]
    if len(lag_dataframe) == 0: return np.nan

    actual_exceedance = (lag_dataframe["target_t_plus_3"] > pm25_threshold).astype(int).values
    predicted_exceedance = (
        (lag_dataframe["lag_0"] > pm25_threshold) | 
        (lag_dataframe["lag_1"] > pm25_threshold) | 
        (lag_dataframe["lag_2"] > pm25_threshold)
    ).astype(int).values
    
    return float(recall_score(actual_exceedance, predicted_exceedance, zero_division=0))

def select_probability_threshold(
    predicted_probabilities, actual_binary_labels, selection_mode="fixed", 
    fixed_probability_threshold=0.35, target_recall_score=None
):
    """目前強制使用 fixed_probability_threshold (0.35)"""
    if selection_mode == "fixed":
        return float(fixed_probability_threshold), None

    threshold_grid = np.linspace(0.01, 0.99, 99)
    evaluation_metrics_rows = []
    
    for candidate_threshold in threshold_grid:
        binary_predictions = (predicted_probabilities >= candidate_threshold).astype(int)
        evaluation_metrics_rows.append((
            candidate_threshold, 
            recall_score(actual_binary_labels, binary_predictions, zero_division=0),
            precision_score(actual_binary_labels, binary_predictions, zero_division=0),
            f1_score(actual_binary_labels, binary_predictions, zero_division=0)
        ))
        
    metrics_dataframe = pd.DataFrame(evaluation_metrics_rows, columns=["probability_threshold", "recall", "precision", "f1_score"])

    if selection_mode == "target_recall_max3":
        if target_recall_score is not None and np.isfinite(target_recall_score):
            feasible_thresholds = metrics_dataframe[metrics_dataframe["recall"] >= target_recall_score]
            if len(feasible_thresholds) > 0:
                best_threshold_row = feasible_thresholds.sort_values(["precision", "f1_score", "probability_threshold"], ascending=[False, False, False]).iloc[0]
                return float(best_threshold_row["probability_threshold"]), metrics_dataframe
                
        best_threshold_row = metrics_dataframe.sort_values(["f1_score", "precision"], ascending=[False, False]).iloc[0]
        return float(best_threshold_row["probability_threshold"]), metrics_dataframe

    raise ValueError("selection_mode must be 'fixed' or 'target_recall_max3'")

特徵工程（Feature Engineering）區塊。負責萃取長格式資料中的 baseline_columns (基礎特徵) 與 wildfire_columns (野火特徵) 並乾淨地分開回傳。

In [4]:
# -------------------------
# Feature Engineering Module
# -------------------------
def append_lagged_features(input_dataframe, target_column_name='target_variable', lag_hours_list=(3, 24, 48)):
    processed_dataframe = input_dataframe.copy()
    for lag_value in lag_hours_list:
        lag_column_name = f'lag_{lag_value}h'
        processed_dataframe[lag_column_name] = processed_dataframe[target_column_name].shift(lag_value)
    return processed_dataframe

def append_temporal_features(input_dataframe):
    processed_dataframe = input_dataframe.copy()
    processed_dataframe['hour_of_day'] = processed_dataframe.index.hour
    processed_dataframe['day_of_week'] = processed_dataframe.index.dayofweek
    processed_dataframe['month_of_year'] = processed_dataframe.index.month
    return processed_dataframe

def prepare_regional_features_from_raw(raw_input_dataframe, region_name):
    regional_dataframe = raw_input_dataframe[raw_input_dataframe['Zone'] == region_name].copy()
    regional_dataframe = regional_dataframe.set_index('Datetime_UTC').sort_index()
    if regional_dataframe.index.tz is not None:
        regional_dataframe.index = regional_dataframe.index.tz_localize(None)

    regional_dataframe['target_variable'] = regional_dataframe['PM25']
    regional_dataframe = append_lagged_features(regional_dataframe, target_column_name='target_variable', lag_hours_list=(3, 24, 48))
    regional_dataframe = append_temporal_features(regional_dataframe)

    baseline_columns = ['lag_3h', 'lag_24h', 'lag_48h', 'hour_of_day', 'day_of_week', 'month_of_year']
    
    wildfire_columns = [
        'fire_count_regional', 'frp_regional_sum', 'hfi_weighted', 
        'fwi_mean', 'fire_count_local', 'frp_local_sum'
    ]
    
    for column_name in wildfire_columns:
        regional_dataframe[column_name] = regional_dataframe[column_name].fillna(0) if column_name in regional_dataframe.columns else 0

    regional_dataframe = regional_dataframe.dropna(subset=['target_variable'] + baseline_columns)
    
    return regional_dataframe, baseline_columns, wildfire_columns

核心建模與推論模組。我將原本 evaluate_classification 中落落長且重複的機率計算邏輯，獨立抽成了一個乾淨的 calculate_exceedance_probabilities 函數。這樣主分類評估函數的職責就變得非常單一且好讀。

In [5]:
# -------------------------
# Model Building & Classification Inference Modules
# -------------------------
def build_dynamic_gam(training_features, training_target, baseline_columns, extra_columns=None):
    if extra_columns is None: extra_columns = []
        
    index_lag3 = training_features.columns.get_loc('lag_3h')
    index_lag24 = training_features.columns.get_loc('lag_24h')
    index_lag48 = training_features.columns.get_loc('lag_48h')
    index_hour = training_features.columns.get_loc('hour_of_day')
    index_dow = training_features.columns.get_loc('day_of_week')
    index_month = training_features.columns.get_loc('month_of_year')
    
    gam_formula = (
        s(index_lag3) + s(index_lag24) + s(index_lag48) +
        te(index_hour, index_dow, n_splines=[12, 7]) +
        te(index_hour, index_month, basis=['cp', 'ps'], n_splines=[12, 6])
    )
    
    for column_name in extra_columns:
        feature_index = training_features.columns.get_loc(column_name)
        gam_formula += s(feature_index, n_splines=10) 
        
    return LinearGAM(gam_formula, lam=0.2454).fit(training_features, training_target)

def fit_calibration_garch_model(calibration_residuals):
    arch_model_instance = arch_model(calibration_residuals, p=1, q=1, vol="Garch", dist="normal", rescale=False)
    fitted_garch_results = arch_model_instance.fit(disp="off")
    garch_parameters = fitted_garch_results.params
    
    return {
        "omega_constant": float(garch_parameters["omega"]),
        "alpha_coefficient": float(garch_parameters["alpha[1]"]),
        "beta_coefficient": float(garch_parameters["beta[1]"]),
        "last_residual_from_training": float(np.asarray(fitted_garch_results.resid)[-1]),
        "last_variance_from_training": float(np.asarray(fitted_garch_results.conditional_volatility)[-1] ** 2)
    }

def calculate_exceedance_probabilities(trained_gam_model, feature_dataframe, actual_target_values, garch_parameters, pm25_physical_threshold):
    """獨立提取的重複邏輯：計算預測平均值、殘差、GARCH標準差，最終輸出超標機率"""
    predicted_mean = trained_gam_model.predict(feature_dataframe)
    prediction_residuals = actual_target_values - predicted_mean
    
    predicted_std_dev = calculate_garch_sigma_manually(prediction_residuals, **garch_parameters)
    
    exceedance_probabilities = 1 - norm.cdf((pm25_physical_threshold - predicted_mean) / (predicted_std_dev + 1e-12))
    return exceedance_probabilities, prediction_residuals

def evaluate_classification(
    trained_gam_model, regional_dataframe, feature_columns_to_use,
    pm25_physical_threshold=15, risk_probability_threshold=0.35,
    garch_calibration_year=2024, testing_year=2025,
    threshold_selection_mode="fixed"
):
    calibration_mask = (regional_dataframe.index.year == garch_calibration_year)
    testing_mask = (regional_dataframe.index.year == testing_year)
    if calibration_mask.sum() == 0 or testing_mask.sum() == 0: return None
    
    calibration_features, calibration_targets = regional_dataframe.loc[calibration_mask, feature_columns_to_use], regional_dataframe.loc[calibration_mask, "target_variable"].values
    testing_features, testing_targets = regional_dataframe.loc[testing_mask, feature_columns_to_use], regional_dataframe.loc[testing_mask, "target_variable"].values
    
    # 透過剛剛獨立出來的函數，先取得校正集的殘差，並擬合 GARCH 參數
    predicted_mean_calibration = trained_gam_model.predict(calibration_features)
    calibration_residuals = calibration_targets - predicted_mean_calibration
    garch_parameters = fit_calibration_garch_model(calibration_residuals)
    
    # 獲得校正集的機率，用於決定 Threshold
    probabilities_calibration, _ = calculate_exceedance_probabilities(
        trained_gam_model, calibration_features, calibration_targets, garch_parameters, pm25_physical_threshold
    )
    
    target_recall = calculate_naive_max3_recall(regional_dataframe['target_variable'], pm25_threshold=pm25_physical_threshold, evaluation_year=garch_calibration_year) if threshold_selection_mode != "fixed" else None
    
    applied_probability_threshold, _ = select_probability_threshold(
        probabilities_calibration, (calibration_targets > pm25_physical_threshold).astype(int), 
        selection_mode=threshold_selection_mode, 
        fixed_probability_threshold=risk_probability_threshold, 
        target_recall_score=target_recall
    )
    
    # 獲得測試集的機率，並計算最終分類指標
    probabilities_testing, _ = calculate_exceedance_probabilities(
        trained_gam_model, testing_features, testing_targets, garch_parameters, pm25_physical_threshold
    )
    
    actual_binary_labels = (testing_targets > pm25_physical_threshold).astype(int)
    predicted_binary_labels = (probabilities_testing >= applied_probability_threshold).astype(int)
    
    return {
        "Alert_Probability_Threshold": applied_probability_threshold,
        "Recall": float(recall_score(actual_binary_labels, predicted_binary_labels, zero_division=0)),
        "Precision": float(precision_score(actual_binary_labels, predicted_binary_labels, zero_division=0)),
        "Accuracy": float(accuracy_score(actual_binary_labels, predicted_binary_labels)),
        "F1": float(f1_score(actual_binary_labels, predicted_binary_labels, zero_division=0))
    }

主程式執行區域。負責執行 AIC 窮舉法 (perform_exhaustive_selection_aic)，程式碼中的變數如 combination 和 candidate_columns 都寫得清楚明白。跑完後直接給出一目了然的報表。

In [6]:
# =========================
# Main Execution Pipeline (Exhaustive Search + Classification Warning)
# =========================
def perform_exhaustive_selection_aic(training_features, training_target, baseline_columns, candidate_columns):
    best_aic_score = float('inf')
    best_feature_combination = []
    
    all_possible_combinations = []
    for count in range(len(candidate_columns) + 1):
        all_possible_combinations.extend(combinations(candidate_columns, count))
        
    print(f"    [Start] Testing all {len(all_possible_combinations)} possible feature combinations...")
    
    for feature_combination in all_possible_combinations:
        feature_combination_list = list(feature_combination)
        testing_gam_model = build_dynamic_gam(
            training_features, training_target, 
            baseline_columns=baseline_columns, extra_columns=feature_combination_list
        )
        current_aic_score = testing_gam_model.statistics_['AIC']
        
        if current_aic_score < best_aic_score:
            best_aic_score = current_aic_score
            best_feature_combination = feature_combination_list
            print(f"    [*] New Best AIC: {best_aic_score:.2f} | Features: {best_feature_combination}")

    print(f"    [Done] Exhaustive Search Finished. Final Best AIC: {best_aic_score:.2f}")
    return best_feature_combination

if __name__ == "__main__":
    dataset_file_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / "CS2_model_input.csv"
    raw_input_dataframe = pd.read_csv(dataset_file_path)
    raw_input_dataframe['Datetime_UTC'] = pd.to_datetime(raw_input_dataframe['Datetime_UTC'])

    experiment_results_list = []
    available_regions_list = raw_input_dataframe['Zone'].unique()

    print("--- Running Model Pipeline (Classification Warning at Threshold 0.35) ---")

    for region_name in available_regions_list:
        print(f"\nProcessing Region: {region_name} ...")
        regional_dataframe, baseline_columns, wildfire_columns = prepare_regional_features_from_raw(raw_input_dataframe, region_name)
        
        training_dataset = regional_dataframe[regional_dataframe.index.year < 2023]
        if len(training_dataset) < 200: continue

        # --- 步驟 A：評估 Baseline 模型 ---
        gam_baseline_model = build_dynamic_gam(
            training_dataset[baseline_columns], training_dataset['target_variable'], 
            baseline_columns=baseline_columns, extra_columns=[]
        )
        metrics_baseline = evaluate_classification(
            gam_baseline_model, regional_dataframe, baseline_columns,
            pm25_physical_threshold=15, risk_probability_threshold=0.35,
            threshold_selection_mode="fixed"
        )
        if metrics_baseline:
            metrics_baseline.update({"Region_Name": region_name, "Model_Type": "1_Baseline"})
            experiment_results_list.append(metrics_baseline)

        # --- 步驟 B：執行 窮舉法 (AIC) 尋找完美特徵組合 ---
        best_wildfire_columns = perform_exhaustive_selection_aic(
            training_dataset[baseline_columns + wildfire_columns], 
            training_dataset['target_variable'], 
            baseline_columns, wildfire_columns
        )

        # --- 步驟 C：評估 最佳特徵組合模型 ---
        best_extended_columns = baseline_columns + best_wildfire_columns
        gam_best_model = build_dynamic_gam(
            training_dataset[best_extended_columns], training_dataset['target_variable'], 
            baseline_columns=baseline_columns, extra_columns=best_wildfire_columns
        )
        metrics_best = evaluate_classification(
            gam_best_model, regional_dataframe, best_extended_columns,
            pm25_physical_threshold=15, risk_probability_threshold=0.35,
            threshold_selection_mode="fixed"
        )
        if metrics_best:
            metrics_best.update({
                "Region_Name": region_name, 
                "Model_Type": f"2_Best_AIC_Model (features: {len(best_wildfire_columns)})"
            })
            experiment_results_list.append(metrics_best)

    # 顯示最終比較報表
    final_classification_report = pd.DataFrame(experiment_results_list).set_index(["Region_Name", "Model_Type"]).sort_index()

    print("\n=== Exhaustive Search (AIC) Final Classification Report ===")
    display(final_classification_report)

--- Running Model Pipeline (Classification Warning at Threshold 0.35) ---

Processing Region: Lower Fraser Valley ...
    [Start] Testing all 64 possible feature combinations...
    [*] New Best AIC: 38120.69 | Features: []
    [*] New Best AIC: 38092.78 | Features: ['fire_count_regional']
    [*] New Best AIC: 38083.47 | Features: ['hfi_weighted']
    [*] New Best AIC: 38066.04 | Features: ['fire_count_local']
    [*] New Best AIC: 38063.99 | Features: ['fire_count_regional', 'fire_count_local']
    [*] New Best AIC: 38062.87 | Features: ['hfi_weighted', 'fire_count_local']
    [*] New Best AIC: 38059.54 | Features: ['fire_count_regional', 'frp_regional_sum', 'fire_count_local']
    [*] New Best AIC: 38059.35 | Features: ['fire_count_regional', 'frp_regional_sum', 'hfi_weighted', 'fire_count_local']
    [*] New Best AIC: 38059.27 | Features: ['fire_count_regional', 'frp_regional_sum', 'fwi_mean', 'fire_count_local']
    [*] New Best AIC: 38058.23 | Features: ['hfi_weighted', 'fwi_mean

C:\Users\User\AppData\Local\Temp\ipykernel_5124\960423631.py:28: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  fitted_garch_results = arch_model_instance.fit(disp="off")


Alert_Probability_Threshold  \
Region_Name         Model_Type                                                    
Lower Fraser Valley 1_Baseline                                             0.35   
                    2_Best_AIC_Model (features: 4)                         0.35   
Southern Interior   1_Baseline                                             0.35   
                    2_Best_AIC_Model (features: 6)                         0.35   

                                                      Recall  Precision  \
Region_Name         Model_Type                                            
Lower Fraser Valley 1_Baseline                      0.717949   0.577320   
                    2_Best_AIC_Model (features: 4)  0.717949   0.595745   
Southern Interior   1_Baseline                      0.771831   0.798834   
                    2_Best_AIC_Model (features: 6)  0.763380   0.763380   

                                                    Accuracy        F1  
Region_Name         Model_Type                                          
Lower Fraser Valley 1_Baseline                      0.992808  0.640000  
                    2_Best_AIC_Model (features: 4)  0.993151  0.651163  
Southern Interior   1_Baseline                      0.982877  0.785100  
                    2_Best_AIC_Model (features: 6)  0.980822  0.763380